# 01 · THEIA 预处理

本 Notebook 是六个 Notebook 中唯一涉及本地 PostgreSQL 恢复/预处理的入口。它会先解析生产配置并检查 graphs、Word2Vec、edge embeddings 与 metadata cache；完整产物存在时会说明原因并跳过。Detection-only 不需要数据库，但首次预处理通常需要已恢复的 DARPA PostgreSQL 数据库。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# 用户参数：THEIA_E3 可改为 THEIA_E5。
PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"
PERSIST_ROOT = DRIVE_ROOT / "artifacts"
DATASET = "THEIA_E3"
PREPROCESS_CONFIG = PROJECT_ROOT / "config/orthrus.yml"
FORCE_PREPROCESS = False
RESTORE_DATABASE = False
DATABASE_DUMP = Path("")  # RESTORE_DATABASE=True 时填写 .dump/.backup 文件。
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
assert DATASET in {"THEIA_E3", "THEIA_E5"}
assert PREPROCESS_CONFIG.is_file(), PREPROCESS_CONFIG

In [ ]:
# 通过生产 config loader 解析真实预处理路径。
src_root = str(PROJECT_ROOT / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
from config import get_runtime_required_args, get_yml_cfg

config_args = get_runtime_required_args(args=[
    DATASET, "--config", str(PREPROCESS_CONFIG), "--artifact-root", str(ARTIFACT_ROOT),
    "--stages", "preprocess", "--skip-tracing",
])
cfg = get_yml_cfg(config_args)
required_paths = {
    "graphs": Path(cfg.graph_construction.build_graphs._graphs_dir),
    "word2vec": Path(cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir),
    "edge_embeddings": Path(cfg.edge_featurization.embed_edges._edge_embeds_dir),
    "metadata": Path(cfg._metadata_dir),
}

def visible_entries(path):
    return [item for item in path.rglob("*") if item.is_file()] if path.is_dir() else []

for label, path in required_paths.items():
    print(f"{label}: {path}（文件数={len(visible_entries(path))}）")
artifacts_complete = all(visible_entries(path) for path in required_paths.values())
print("完整预处理产物：", artifacts_complete)

In [ ]:
# 可选 PostgreSQL 检查/恢复。密码只从环境变量读取，命令与输出均不会打印它。
if RESTORE_DATABASE:
    required = ["ORTHRUS_DB_HOST", "ORTHRUS_DB_PORT", "ORTHRUS_DB_USER", "ORTHRUS_DB_PASSWORD"]
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise RuntimeError("缺少数据库环境变量：" + ", ".join(missing))
    if not DATABASE_DUMP.is_file():
        raise FileNotFoundError(DATABASE_DUMP)
    db_name = (cfg.dataset.database_all_file if cfg.graph_construction.build_graphs.use_all_files else cfg.dataset.database)
    pg_env = os.environ.copy()
    pg_env["PGPASSWORD"] = os.environ["ORTHRUS_DB_PASSWORD"]
    subprocess.run([
        "pg_isready", "-h", os.environ["ORTHRUS_DB_HOST"],
        "-p", os.environ["ORTHRUS_DB_PORT"], "-U", os.environ["ORTHRUS_DB_USER"],
    ], env=pg_env, check=True)
    subprocess.run([
        "createdb", "-h", os.environ["ORTHRUS_DB_HOST"], "-p", os.environ["ORTHRUS_DB_PORT"],
        "-U", os.environ["ORTHRUS_DB_USER"], db_name,
    ], env=pg_env, check=False)
    subprocess.run([
        "pg_restore", "--clean", "--if-exists", "--no-owner",
        "-h", os.environ["ORTHRUS_DB_HOST"], "-p", os.environ["ORTHRUS_DB_PORT"],
        "-U", os.environ["ORTHRUS_DB_USER"], "-d", db_name, str(DATABASE_DUMP),
    ], env=pg_env, check=True)
else:
    print("RESTORE_DATABASE=False：不触碰 PostgreSQL；已有预处理产物可直接用于 detection-only。")

In [ ]:
if artifacts_complete and not FORCE_PREPROCESS:
    print("检测到 graphs、Word2Vec、edge embeddings 与 metadata cache，跳过预处理。")
    print("如需明确重跑，请先评估成本，再将 FORCE_PREPROCESS=True；Notebook 不会静默覆盖。")
else:
    reason = "FORCE_PREPROCESS=True" if FORCE_PREPROCESS else "预处理产物不完整"
    print("开始预处理，原因：", reason)
    command = [
        sys.executable, str(PROJECT_ROOT / "src/orthrus.py"), DATASET,
        "--config", str(PREPROCESS_CONFIG), "--stages", "preprocess",
        "--skip-tracing", "--artifact-root", str(ARTIFACT_ROOT),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

missing_after = {label: str(path) for label, path in required_paths.items() if not visible_entries(path)}
if missing_after:
    raise RuntimeError("预处理后仍缺少产物：" + repr(missing_after))
print("预处理生成物验证通过。")

In [ ]:
# 导出 metadata cache，并在需要时将整个 artifact tree 持久化。
metadata_export = ARTIFACT_ROOT / "metadata_exports" / DATASET
shutil.copytree(required_paths["metadata"], metadata_export, dirs_exist_ok=True)
print("metadata 已导出到：", metadata_export)

# 若运行目录不是 Drive，则复制到持久化根；相同路径时无需重复复制。
if ARTIFACT_ROOT.resolve() == PERSIST_ROOT.resolve():
    print("Artifact 已直接位于 Drive：", ARTIFACT_ROOT)
else:
    PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(ARTIFACT_ROOT, PERSIST_ROOT, dirs_exist_ok=True)
    print("已持久化到：", PERSIST_ROOT)
for label, path in required_paths.items():
    print(label, "OK", path)